# Shared p(True) for reflexive methods: mode comparison (R1 + R2)

**Authors:** Eylül (R1, zero-shot p(True)) and Elçin (R2, brainstorm p(True))

**Goal:** one shared `common/reflexive.py` for both methods, so that the **only** difference
between zero-shot and brainstorm p(True) is the brainstorm line. Two ways of reading the model's
verdict are compared on the shared sanity test (`common/sanity.py`), and one is chosen as the
main mode for both methods.

**Background:** R1 (token diagnosis, Day 1) and R2 (Day 1–2) found the same problem: the models
answer with the letter (`(A`, `A`) rather than with "True". We solved it in two different ways:

- **chat_groups** (R1): chat template; the next token is compared over the groups
  True / A / (A versus False / B / (B (only strings that are a single token for the tokenizer).
- **plain_ab** (R2): no chat template; the prompt ends with `" ("`, so the next token is almost
  forced to be `A` or `B`; only A versus B is compared.

Both give a normalized p(True) = P(True group) / (P(True group) + P(False group)).
**Coverage** = P(True group) + P(False group) shows how much of the model's probability the
normalized value is built on.

## 1. Shared prompt (agreed before any runs)

    Question: {question}
    Here are some ideas that were brainstormed: {s1, s2, ...}   <- only with brainstorm
    Possible answer: {answer}
    Is the possible answer:
     (A) True
     (B) False
    The possible answer is:                                      <- plain_ab appends " ("

- No leading spaces, except before `(A)` and `(B)`.
- The brainstorm line is only added when samples are given; samples are joined with `", "`.
- The brainstorm line sits between `Question` and `Possible answer`.
- Based on R2's prompt, so R2's earlier plain_ab results can be used to check the shared code.
- Note: R1's earlier numbers (Days 1–3) used a slightly different prompt (leading spaces),
  so they are not directly comparable to the results in this notebook.

## 2. Decision rule (fixed before the comparison)

1. **Main criterion:** mean AUROC **without brainstorm** over Qwen2.5-0.5B and Qwen2.5-1.5B.
2. **Guard:** a mode is ruled out if its **with-brainstorm** AUROC on Qwen 0.5B or 1.5B is more
   than 0.05 below the other mode's. (With brainstorm the sanity test is saturated, ~1.00, so it
   is only used to catch a mode that clearly breaks.)
3. If the main-criterion difference is **< 0.05** → the mode with higher **mean coverage** wins.
4. If still tied → **plain_ab** (simpler: no chat template, only two tokens).

**Why these choices**
- AUROC measures what the method is for (separating correct from wrong answers) and only uses the
  ranking, so it is fair between modes with different scales.
- SmolLM2-135M is excluded from the decision: it is at chance level in both modes.
- The sanity test has only 20 answers (10 correct, 10 wrong), so differences below 0.05 AUROC
  are within noise.
- Coverage as tie-breaker: a normalized p(True) built on more probability mass is more reliable.

## 3. Setup

In [1]:
import sys, json, gc, importlib
from pathlib import Path
import numpy as np
import torch
from probly.representer.sampler.huggingface import load_model

LLM_DIR = Path.cwd().parent              # experiments/llm-probly
sys.path.insert(0, str(LLM_DIR))
SANITY_DIR = LLM_DIR / "results" / "sanity"

print("LLM_DIR:", LLM_DIR)
print("common/:", sorted(p.name for p in (LLM_DIR / "common").glob("*.py")))
print("sanity samples:", sorted(str(p.relative_to(LLM_DIR))
                                for p in (LLM_DIR / "data").glob("*/sanity_samples.json")))

LLM_DIR: /Users/eylulkucukbulut/Projects/llm-probly/experiments/llm-probly
common/: ['__init__.py', 'correctness.py', 'data.py', 'evaluate.py', 'io.py', 'sanity.py', 'summary.py']
sanity samples: ['data/qwen0.5b/sanity_samples.json', 'data/qwen1.5b/sanity_samples.json', 'data/smollm135m/sanity_samples.json']


## 4. Quick test of common/reflexive.py

One question (Paris = correct, Berlin = wrong) on Qwen2.5-0.5B, both modes, plus the raw baseline.
Expected: p_true higher for Paris than for Berlin in both modes; plain_ab coverage much higher
than the 0.02–0.08 R1 measured without chat template on Day 1.

In [2]:
import common.reflexive as rx
importlib.reload(rx)

model, tok = load_model("Qwen/Qwen2.5-0.5B-Instruct")
q = "What is the capital of France?"
for mode in ["chat_groups", "plain_ab"]:
    for ans in ["Paris", "Berlin"]:
        d = rx.p_true_details(model, tok, q, ans, mode=mode)
        print(f"{mode:12s} {ans:7s} p_true={d['p_true']:.3f}  coverage={d['coverage']:.3f}")
for ans in ["Paris", "Berlin"]:
    print(f"{'raw logit':12s} {ans:7s} true_logit={rx.true_logit(model, tok, q, ans):.2f}")
print("\nprompt (plain_ab):\n" + rx.build_prompt(tok, q, "Paris", mode="plain_ab"))
del model, tok; gc.collect()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

chat_groups  Paris   p_true=0.879  coverage=0.677
chat_groups  Berlin  p_true=0.618  coverage=0.626
plain_ab     Paris   p_true=0.963  coverage=1.000
plain_ab     Berlin  p_true=0.245  coverage=1.000
raw logit    Paris   true_logit=16.62
raw logit    Berlin  true_logit=15.56

prompt (plain_ab):
Question: What is the capital of France?
Possible answer: Paris
Is the possible answer:
 (A) True
 (B) False
The possible answer is: (


2571

## 5. Mode comparison on the sanity test

3 models × 2 modes × with/without brainstorm, using R2's `run_sanity()` (10 questions,
correct vs wrong answer each = 20 answers). Brainstorm samples: `data/<model>/sanity_samples.json`.
Coverage is averaged over the same 20 answers.

Check: the plain_ab rows should reproduce R2's earlier results (same prompt), i.e. about
0.37 / 0.755 / 0.99 without and 0.34 / 1.00 / 1.00 with brainstorm (SmolLM / Qwen 0.5B / 1.5B).

In [3]:
from common.sanity import run_sanity, save_sanity

KEYS = {"HuggingFaceTB/SmolLM2-135M-Instruct": "smollm135m",
        "Qwen/Qwen2.5-0.5B-Instruct": "qwen0.5b",
        "Qwen/Qwen2.5-1.5B-Instruct": "qwen1.5b"}

table = []
for name, key in KEYS.items():
    samples = json.loads((LLM_DIR / "data" / key / "sanity_samples.json").read_text())
    model, tok = load_model(name)
    for mode in ["chat_groups", "plain_ab"]:
        for bs in [False, True]:
            covs = []
            def fn(q, a, s):
                d = rx.p_true_details(model, tok, q, a, brainstorm=s, mode=mode)
                covs.append(d["coverage"])
                return d["p_true"]
            res = run_sanity(fn, samples=samples if bs else None)
            res["mean_coverage"] = float(np.mean(covs))
            method = f"p_true_{mode}_{'brainstorm' if bs else 'zeroshot'}"
            save_sanity(res, method, name, folder=SANITY_DIR)
            table.append({"model": key, "mode": mode, "brainstorm": bs,
                          "auroc": res["auroc"], "gap": res["gap"], "coverage": res["mean_coverage"]})
            print(f"{key:11s} {mode:12s} brainstorm={str(bs):5s} AUROC={res['auroc']:.3f}  "
                  f"gap={res['gap']:+.3f}  coverage={res['mean_coverage']:.3f}")
    del model, tok
    gc.collect();

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

smollm135m  chat_groups  brainstorm=False AUROC=0.510  gap=-0.000  coverage=0.251
smollm135m  chat_groups  brainstorm=True  AUROC=0.510  gap=-0.008  coverage=0.273
smollm135m  plain_ab     brainstorm=False AUROC=0.290  gap=-0.013  coverage=0.720
smollm135m  plain_ab     brainstorm=True  AUROC=0.370  gap=-0.037  coverage=0.676


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

qwen0.5b    chat_groups  brainstorm=False AUROC=0.670  gap=+0.060  coverage=0.612
qwen0.5b    chat_groups  brainstorm=True  AUROC=0.940  gap=+0.310  coverage=0.491
qwen0.5b    plain_ab     brainstorm=False AUROC=0.755  gap=+0.145  coverage=1.000
qwen0.5b    plain_ab     brainstorm=True  AUROC=1.000  gap=+0.515  coverage=1.000


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

qwen1.5b    chat_groups  brainstorm=False AUROC=0.990  gap=+0.827  coverage=0.962
qwen1.5b    chat_groups  brainstorm=True  AUROC=1.000  gap=+0.815  coverage=0.910
qwen1.5b    plain_ab     brainstorm=False AUROC=0.990  gap=+0.782  coverage=1.000
qwen1.5b    plain_ab     brainstorm=True  AUROC=1.000  gap=+0.834  coverage=1.000


In [4]:
import pandas as pd
df = pd.DataFrame(table)
display(df.round(3))

qwen = df[df.model.isin(["qwen0.5b", "qwen1.5b"])]
main = qwen[~qwen.brainstorm].groupby("mode")[["auroc", "coverage"]].mean()
guard = qwen[qwen.brainstorm].pivot(index="model", columns="mode", values="auroc")
print("\nMain criterion (mean AUROC without brainstorm, Qwen 0.5B + 1.5B):\n", main.round(3))
print("\nGuard (AUROC with brainstorm):\n", guard.round(3))

,model,mode,brainstorm,auroc,gap,coverage
0,smollm135m,chat_groups,False,0.510,-0.000,0.251
1,smollm135m,chat_groups,True,0.510,-0.008,0.273
2,smollm135m,plain_ab,False,0.290,-0.013,0.720
3,smollm135m,plain_ab,True,0.370,-0.037,0.676
4,qwen0.5b,chat_groups,False,0.670,0.060,0.612
5,qwen0.5b,chat_groups,True,0.940,0.310,0.491
6,qwen0.5b,plain_ab,False,0.755,0.145,1.000
7,qwen0.5b,plain_ab,True,1.000,0.515,1.000
8,qwen1.5b,chat_groups,False,0.990,0.827,0.962
9,qwen1.5b,chat_groups,True,1.000,0.815,0.910



Main criterion (mean AUROC without brainstorm, Qwen 0.5B + 1.5B):
              auroc  coverage
mode                        
chat_groups  0.830     0.787
plain_ab     0.872     1.000

Guard (AUROC with brainstorm):
 mode      chat_groups  plain_ab
model                          
qwen0.5b         0.94       1.0
qwen1.5b         1.00       1.0


## 6. Decision

| Rule | chat_groups | plain_ab | Result |
|---|---|---|---|
| 1. Mean AUROC without brainstorm (Qwen 0.5B + 1.5B) | 0.830 | 0.873 | difference 0.043 < 0.05 → not decisive |
| 2. Guard: AUROC with brainstorm, Qwen 0.5B | 0.940 | 1.000 | chat_groups 0.06 lower → **ruled out** |
| 2. Guard: AUROC with brainstorm, Qwen 1.5B | 1.000 | 1.000 | equal |
| 3. Mean coverage without brainstorm (Qwen) | 0.787 | 1.000 | plain_ab higher (would also decide) |

**Main mode: plain_ab.** The guard rules out chat_groups (rule 2), and coverage would choose
plain_ab as well (rule 3), so the decision does not depend on a single rule.

**Observations**
- plain_ab reproduces R2's earlier Qwen results exactly (AUROC and gap), which confirms that the
  shared code computes the same as R2's notebook.
- SmolLM2-135M differs from R2's earlier runs (0.29 vs 0.37 without, 0.37 vs 0.34 with brainstorm).
  Its gap is ~0 in all settings, so tiny numerical differences reorder the ranking; all values
  are at chance level and SmolLM is excluded from the decision.
- Brainstorm helps clearly on Qwen 0.5B in both modes (0.67 → 0.94 and 0.755 → 1.00).
- R1's finding (letter answers, coverage) is what makes the comparison possible: chat_groups
  covers only ~0.6 of the probability on Qwen 0.5B, plain_ab ~1.0.
